# 07 — Évaluation des réponses : fidélité, pertinence, citations et refus

Une réponse peut reproduire fidèlement un document tout en répondant mal à la question.
Ce notebook distingue la qualité du contenu, la pertinence de la source et la décision
de répondre ou de refuser.

**Objectif :** analyser les trois réponses imposées par l’énoncé, puis mesurer les
résultats de notre assistant extractif sur les questions annotées.

Le parcours commence par une revue métier des trois réponses et de leurs corrections.
Il évalue ensuite les réponses produites avec les trois retrievers, présente les taux
et examine les erreurs individuelles. Les jugements humains et les contrôles
mécaniques sont explicitement séparés.


In [1]:
from contextiq.ingestion.loader import DataLoader
from contextiq.settings import Settings

settings = Settings.load()
dataset = DataLoader().load()
documents_by_id = {document.id: document for document in dataset.documents}

08:58:08 | INFO     | Début de l’opération | {'operation': 'contextiq.settings.Settings.load', 'trace_id': '76af43215e95', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:08 | INFO     | Opération terminée | {'operation': 'contextiq.settings.Settings.load', 'trace_id': '76af43215e95', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 4.636}
08:58:08 | INFO     | Début de l’opération | {'operation': 'contextiq.ingestion.loader.DataLoader.__init__', 'trace_id': '19cd98b1d2aa', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:08 | INFO     | Opération terminée | {'operation': 'contextiq.ingestion.loader.DataLoader.__init__', 'trace_id': '19cd98b1d2aa', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 0.281}
08:58:08 | INFO     | Début de l’opération | {'operation': 'contextiq.ingestion.loader.DataLoader.load',

## 1. Définir les critères de la revue métier

La **fidélité** vérifie si les affirmations respectent le document de référence.
La **pertinence thématique** indique si la réponse traite le sujet de la question.
La **justesse métier** vérifie si elle permet au conseiller de prendre une décision correcte.
La **citation** doit désigner une source qui appuie l’affirmation.

Les trois verdicts suivants résultent d’une lecture des sources. Ils ne sont calculés
ni par un LLM ni par les métriques de correspondance exacte de notre générateur.
Une réponse peut être sur le sujet et néanmoins dangereuse pour la décision métier.


### Cas 1 — Frais de résiliation d’un contrat annuel

**Réponse fournie :** « Les frais sont de 50 % des mensualités restantes, sans plafond. »

**Fidélité : non.** Le taux de 50 % est correct, mais l’absence de plafond contredit D2,
qui limite les frais à 150 euros. Cette erreur peut conduire à annoncer des frais trop élevés.

**Pertinence :** la réponse est sur le sujet, mais elle est incorrecte et inexploitable
en l’état pour le conseiller. Aucune source n’est citée dans la réponse fournie.

**Correction :** les frais correspondent à 50 % des mensualités restantes, plafonnés
à 150 euros. [D2]

Le passage de référence est affiché ci-dessous.


In [2]:
print("SOURCE D2 — FRAIS DE RÉSILIATION")
print("-" * 80)
print(documents_by_id["D2"].text)

SOURCE D2 — FRAIS DE RÉSILIATION
--------------------------------------------------------------------------------
La résiliation d'un contrat annuel avant son terme entraîne des frais égaux à 50 % des mensualités restantes, plafonnés à 150 euros.


### Cas 2 — Remise pour un client de six mois d’ancienneté

**Réponse fournie :** « Oui, une remise de -30 % pendant 3 mois est possible. »

**Fidélité : non.** D5 réserve cette offre aux clients ayant **plus de 12 mois**
d’ancienneté et sans retard de paiement. Un client de six mois n’est donc pas éligible.
La réponse reprend le montant et la durée tout en omettant la condition décisive.

**Pertinence :** elle traite bien la demande de remise, mais sa recommandation métier
est fausse. Aucune source n’est citée dans la réponse fournie.

**Correction :** non, cette offre n’est pas applicable à six mois d’ancienneté. Elle exige
plus de 12 mois d’ancienneté et l’absence de retard de paiement. [D5]


In [3]:
print("SOURCE D5 — CONDITIONS DE L’OFFRE DE RÉTENTION")
print("-" * 80)
print(documents_by_id["D5"].text)

SOURCE D5 — CONDITIONS DE L’OFFRE DE RÉTENTION
--------------------------------------------------------------------------------
Une offre de rétention de 3 mois à -30 % peut être proposée uniquement aux clients ayant plus de 12 mois d'ancienneté et sans retard de paiement.


### Cas 3 — Délai de préavis d’un contrat mensuel

**Réponse fournie :** « Le préavis est de 10 jours et sans pénalité (source : D1). »

**Fidélité : oui.** D1 confirme le délai de 10 jours et l’absence de pénalité.
**Pertinence : oui.** La réponse traite directement la question du contrat mensuel.
**Citation : correcte.** D1 justifie les deux affirmations.

La réponse peut être conservée en précisant « pour un contrat mensuel » pour éviter
une généralisation à d’autres contrats.


In [4]:
print("SOURCE D1 — PRÉAVIS MENSUEL")
print("-" * 80)
print(documents_by_id["D1"].text)

SOURCE D1 — PRÉAVIS MENSUEL
--------------------------------------------------------------------------------
Un client peut résilier son contrat mensuel à tout moment avec un préavis de 10 jours. Aucune pénalité n'est appliquée.


### Synthèse des trois réponses

| Réponse | Fidèle | Sur le sujet | Correcte pour la décision métier | Citation présente et correcte |
|---|---|---|---|---|
| Frais annuels sans plafond | Non | Oui | Non : plafond de 150 euros omis | Non |
| Remise à six mois | Non | Oui | Non : condition d’ancienneté non respectée | Non |
| Préavis mensuel de 10 jours | Oui | Oui | Oui | Oui, D1 |

Une formulation plausible et des chiffres partiellement corrects ne suffisent pas :
les plafonds et conditions d’éligibilité doivent être vérifiés au même titre que les montants.


## 2. Définir les contrôles des réponses extractives

Notre générateur recopie un document entier et ajoute sa citation. Il permet des
contrôles automatiques adaptés à ce format :

- **Source attendue retrouvée :** la source restituée appartient aux annotations de la question. C’est un indicateur de pertinence, pas un jugement sémantique.
- **Fidélité verbatim :** le passage et sa source déclarée correspondent au document original. La citation est contrôlée séparément.
- **Citation valide :** l’identifiant final correspond à l’unique source déclarée et existe dans la base. Cela ne prouve pas que la source répond à la question.
- **Refus correct :** aucune source n’est attendue, aucune n’est exposée et le message standard de refus est restitué.
- **Succès conjoint :** une question avec source reçoit un extrait fidèle, une citation valide et une source attendue ; une question hors périmètre reçoit un refus correct.

Les refus ont des valeurs « non applicables » pour la fidélité de l’extrait et la citation,
plutôt qu’un score zéro artificiel. Le contrôle verbatim n’est pas un évaluateur de
réponses libres ou reformulées : les trois cas précédents nécessitent la revue métier.


In [5]:
import logging

# Message macOS concernant une redirection multiprocessus inutilisée ici.
# Les autres avertissements et erreurs restent visibles.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(
    lambda record: record.getMessage() != "NOTE: Redirects are currently not supported in MacOs."
)

In [6]:
from contextiq.evaluation.answer_metrics import ExtractiveAnswerEvaluator
from contextiq.generation.extractive_answer import ExtractiveAnswerGenerator
from contextiq.retrieval.bm25 import BM25Retriever
from contextiq.retrieval.embeddings import EmbeddingRetriever
from contextiq.retrieval.tfidf import TfidfRetriever
from contextiq.safety.abstention import AbstentionPolicy

retrievers = {
    "TF-IDF": TfidfRetriever(dataset.documents),
    "BM25": BM25Retriever(dataset.documents, **settings.retrieval.bm25.model_dump()),
    "Embeddings": EmbeddingRetriever(
        dataset.documents, **settings.retrieval.embeddings.model_dump()
    ),
}
generator = ExtractiveAnswerGenerator()
evaluator = ExtractiveAnswerEvaluator(dataset.documents)

08:58:15 | INFO     | Début de l’opération | {'operation': 'contextiq.retrieval.tfidf.TfidfRetriever.__init__', 'trace_id': 'acfef384ea87', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:15 | INFO     | Début de l’opération | {'operation': 'contextiq.ingestion.preprocessing.TextPreprocessor.tokenize', 'trace_id': 'acfef384ea87', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:15 | INFO     | Opération terminée | {'operation': 'contextiq.ingestion.preprocessing.TextPreprocessor.tokenize', 'trace_id': 'acfef384ea87', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 1.55, 'result_count': 21}
08:58:15 | INFO     | Début de l’opération | {'operation': 'contextiq.ingestion.preprocessing.TextPreprocessor.tokenize', 'trace_id': 'acfef384ea87', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:15 | INFO     | Opération ter

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

08:58:17 | INFO     | Opération terminée | {'operation': 'contextiq.retrieval.embeddings.EmbeddingRetriever.__init__', 'trace_id': '3a71289421a8', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 1390.833, 'local_files_only': True, 'document_count': 6}


## 3. Produire puis évaluer les réponses

Pour chaque méthode, les questions traversent la recherche, l’abstention et le contrôle
de sécurité. L’annotation est consultée uniquement après la production de la réponse.
Les seuils restent ceux des configurations, sans optimisation sur les scores obtenus.

L’évaluateur conserve sa propre copie des documents originaux : une réponse ne peut
pas servir de seule référence pour valider sa propre fidélité.


In [7]:
evaluations = {}
summaries = {}

for method, retriever in retrievers.items():
    policy = AbstentionPolicy(threshold=settings.retrieval.abstention_thresholds[method])
    cases = []
    for question in dataset.questions:
        results = retriever.retrieve(question.q, top_k=settings.retrieval.top_k)
        answer = generator.generate(question.q, policy.decide(results))
        cases.append(evaluator.evaluate(answer, question))
    evaluations[method] = cases
    summaries[method] = evaluator.summarize(cases)

08:58:17 | INFO     | Début de l’opération | {'operation': 'contextiq.retrieval.tfidf.TfidfRetriever.retrieve', 'trace_id': '6580b95a52d1', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'top_k': 3}
08:58:17 | INFO     | Début de l’opération | {'operation': 'contextiq.ingestion.preprocessing.TextPreprocessor.tokenize', 'trace_id': '6580b95a52d1', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application'}
08:58:17 | INFO     | Opération terminée | {'operation': 'contextiq.ingestion.preprocessing.TextPreprocessor.tokenize', 'trace_id': '6580b95a52d1', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 0.975, 'result_count': 9}
08:58:17 | INFO     | Opération terminée | {'operation': 'contextiq.retrieval.tfidf.TfidfRetriever.retrieve', 'trace_id': '6580b95a52d1', 'application': 'contextiq', 'session_id': '50a423b6bcef', 'context': 'application', 'duration_ms': 9.128, 'top_k': 3, '

## 4. Comparer les résultats et leurs dénominateurs

Le succès conjoint et le taux de réponse portent sur toutes les questions.
La fidélité et les citations portent uniquement sur les extraits produits.
Le taux de source attendue porte sur les questions avec source ayant reçu un extrait.
Les faux refus sont rapportés aux questions avec source ; les refus corrects aux questions
hors périmètre. « Non applicable » indique un groupe vide.

Ces dénominateurs sont essentiels : une méthode qui refuse beaucoup peut conserver une
bonne fidélité sur les rares réponses produites tout en répondant mal au besoin global.


In [8]:
from IPython.display import Markdown, display

metrics = {
    "Taux de réponse — toutes les questions": "answer_rate",
    "Succès conjoint — toutes les questions": "success_rate",
    "Source attendue — extraits sur questions avec source": "relevant_source_rate",
    "Fidélité verbatim — extraits produits": "faithfulness_rate",
    "Citation valide — extraits produits": "citation_validity_rate",
    "Faux refus — questions avec source": "false_refusal_rate",
    "Refus correct — questions hors périmètre": "correct_refusal_rate",
}
rows = ["| Critère | " + " | ".join(summaries) + " |", "|---|" + "---:|" * len(summaries)]
for label, field in metrics.items():
    values = []
    for summary in summaries.values():
        value = getattr(summary, field)
        values.append(f"{value:.1%}" if value is not None else "Non applicable")
    rows.append("| " + label + " | " + " | ".join(values) + " |")
display(Markdown("\n".join(rows)))

for method, summary in summaries.items():
    print(
        f"{method} : {summary.answered_count}/{summary.question_count} extraits produits ; "
        f"{summary.in_scope_count} questions avec source, {summary.out_of_scope_count} hors périmètre."
    )

| Critère | TF-IDF | BM25 | Embeddings |
|---|---:|---:|---:|
| Taux de réponse — toutes les questions | 66.7% | 66.7% | 66.7% |
| Succès conjoint — toutes les questions | 33.3% | 33.3% | 66.7% |
| Source attendue — extraits sur questions avec source | 66.7% | 66.7% | 75.0% |
| Fidélité verbatim — extraits produits | 100.0% | 100.0% | 100.0% |
| Citation valide — extraits produits | 100.0% | 100.0% | 100.0% |
| Faux refus — questions avec source | 40.0% | 40.0% | 20.0% |
| Refus correct — questions hors périmètre | 0.0% | 0.0% | 100.0% |

TF-IDF : 4/6 extraits produits ; 5 questions avec source, 1 hors périmètre.
BM25 : 4/6 extraits produits ; 5 questions avec source, 1 hors périmètre.
Embeddings : 4/6 extraits produits ; 5 questions avec source, 1 hors périmètre.


## 5. Lire les erreurs au niveau des réponses

Le détail sépare la décision de répondre, la source attendue, la fidélité et la citation.
Un extrait fidèle avec une citation valide peut échouer parce que le document choisi
ne répond pas à la question. Une question pertinente refusée échoue également au
critère conjoint, même si aucun texte incorrect n’a été produit.


In [9]:
method = "Embeddings"
labels = {True: "Oui", False: "Non", None: "Non applicable"}

for number, case in enumerate(evaluations[method], start=1):
    print("\n" + "=" * 80)
    print(f"{number}. QUESTION : {case.question.q}")
    print("-" * 80)
    print("Résultat :", "EXTRAIT" if case.answer.answered else "REFUS")
    print(case.answer.text)
    print()
    print("Sources attendues  :", ", ".join(case.question.gold_docs) or "Hors périmètre")
    print("Source attendue citée :", labels[case.reference_hit])
    print("Passage fidèle       :", labels[case.verbatim_faithful])
    print("Citation valide      :", labels[case.citation_valid])
    print("Décision de périmètre correcte :", labels[case.scope_decision_correct])
    print("Refus correct        :", labels[case.refusal_correct])
    print("Succès conjoint      :", labels[case.successful])
    print("=" * 80)


1. QUESTION : Combien de temps de préavis pour un contrat mensuel ?
--------------------------------------------------------------------------------
Résultat : EXTRAIT
Un client peut résilier son contrat mensuel à tout moment avec un préavis de 10 jours. Aucune pénalité n'est appliquée. [D1]

Sources attendues  : D1
Source attendue citée : Oui
Passage fidèle       : Oui
Citation valide      : Oui
Décision de périmètre correcte : Oui
Refus correct        : Non applicable
Succès conjoint      : Oui

2. QUESTION : Quels frais si je quitte mon abonnement annuel avant la fin ?
--------------------------------------------------------------------------------
Résultat : EXTRAIT
La résiliation d'un contrat annuel avant son terme entraîne des frais égaux à 50 % des mensualités restantes, plafonnés à 150 euros. [D2]

Sources attendues  : D2
Source attendue citée : Oui
Passage fidèle       : Oui
Citation valide      : Oui
Décision de périmètre correcte : Oui
Refus correct        : Non applicable


## Portée des résultats

Ces contrôles évaluent notre format extractif. Ils ne démontrent pas que la source
annotée suffit à répondre dans tous les cas, que les annotations sont exhaustives,
ni que l’assistant sait raisonner sur une situation client. Un refus pour motif de
sécurité doit aussi être étudié avec des annotations spécifiques, distinctes du périmètre.

Une fidélité de 100 % est attendue pour une copie correcte des sources et ne constitue
pas à elle seule une preuve d’efficacité métier. Le succès conjoint reste limité par
le retrieval et les seuils d’abstention. Le petit nombre de questions, dont une seule
hors périmètre, rend ces proportions descriptives.

La revue manuelle des réponses de l’énoncé complète cette mesure : elle révèle des
contradictions et omissions que la simple présence d’une citation ne permet pas de juger.
Le protocole de validation indépendante et le bilan détaillé sont accessibles ci-dessous.
La constitution des jeux enrichis, leur évaluation et le suivi en production restent à réaliser.


## Synthèse et cadre d’évaluation

Le [rapport de résultats](../reports/evaluation_report.md) rassemble les mesures,
les décisions d’abstention et les erreurs de réponse observées. Le
[protocole complet](../reports/evaluation_protocol.md) précise comment enrichir
les cas et conduire une validation indépendante. Les chiffres du rapport décrivent
une exécution exploratoire figée, dont les paramètres et les résultats détaillés sont
conservés dans un instantané associé.
